# Tutorial 12 -- Qubit Ramsey T2*

Build a Ramsey sequence from two `x90` pulses, add dephasing noise, and fit both the detuning and `T2*` from the fringe envelope.

**Prerequisites.** Tutorials 04, 10, and 11 are recommended first.


## 1. Goal

We will implement a textbook Ramsey sequence and extract the detuning and dephasing time from the simulated fringe pattern.


## 2. Physical Background

A Ramsey experiment uses two `pi/2` pulses separated by free evolution. The fringe frequency encodes the residual detuning, while the envelope decays on the `T2*` timescale.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
t1_true = 30.0 * us
t2_star_true = 8.0 * us
detuning = 2.0 * np.pi * 0.6e6
delays_us = np.linspace(0.0, 16.0, 41)
rotation_duration = 30.0 * ns
rotation_sigma_fraction = 0.18
dt = 4.0 * ns


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
frame = FrameSpec(omega_q_frame=model.omega_q)
tphi_true = pure_dephasing_time_from_t1_t2(t1_s=t1_true, t2_s=t2_star_true)
noise = NoiseSpec(t1=t1_true, tphi=tphi_true)
base_pulses, _, _ = build_rotation_pulse(
    RotationGate(index=0, name="x90", theta=np.pi / 2.0, phi=0.0),
    {"duration_rotation_s": rotation_duration, "rotation_sigma_fraction": rotation_sigma_fraction},
)
base_x90 = base_pulses[0]


## 6. Pulse / Sequence Construction


In [ ]:
delays_s = delays_us * us
responses = []
for delay_s in delays_s:
    first_pulse = Pulse(
        channel=base_x90.channel,
        t0=0.0,
        duration=base_x90.duration,
        envelope=base_x90.envelope,
        amp=base_x90.amp,
        carrier=base_x90.carrier,
        phase=0.0,
        label="ramsey_x90_a",
    )
    second_pulse = Pulse(
        channel=base_x90.channel,
        t0=rotation_duration + delay_s,
        duration=base_x90.duration,
        envelope=base_x90.envelope,
        amp=base_x90.amp,
        carrier=base_x90.carrier,
        phase=float(detuning * delay_s),
        label="ramsey_x90_b",
    )
    t_end = 2.0 * rotation_duration + delay_s + dt
    compiled = SequenceCompiler(dt=dt).compile([first_pulse, second_pulse], t_end=t_end)
    result = simulate_sequence(
        model,
        compiled,
        model.basis_state(0, 0),
        {"qubit": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt),
        noise=noise,
    )
    responses.append(final_expectation(result, "P_e"))
responses = np.asarray(responses, dtype=float)


## 7. Running the Simulation


In [ ]:
fit = fit_ramsey_signal(delays_s, responses, p0=(detuning, t2_star_true, 0.5, 0.5, 0.0))
print(f"True detuning / 2pi = {detuning / (2.0 * np.pi * 1.0e6):.3f} MHz")
print(f"Fitted detuning / 2pi = {fit.parameters['detuning'] / (2.0 * np.pi * 1.0e6):.3f} MHz")
print(f"True T2* = {t2_star_true / us:.3f} us")
print(f"Fitted T2* = {fit.parameters['t2_star'] / us:.3f} us")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(delays_us, responses, "o", label="simulation")
ax.plot(delays_us, fit.model_y, "-", label="fit")
ax.set_xlabel("Free-evolution delay [us]")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Ramsey fringe with dephasing")
ax.legend()
plt.show()


## 9. Physical Interpretation

The oscillation frequency reflects the residual phase advance between the two `x90` pulses, while the envelope is set by the combination of energy relaxation and pure dephasing. This is why Ramsey is the natural place to estimate `T2*`.


## 10. Exercises / Next Steps

- Set `detuning = 0` and confirm that the fringes collapse into a pure decay envelope.
- Increase the dephasing rate while keeping `T1` fixed to see how `T2*` changes.
- Continue to Tutorial 13 for a spin-echo sequence that mitigates static phase accumulation.
